#### Name: Blessing Adeniji
#### Degree: MSc Artifical Intelligence Online
#### Capstone Project: AI-Generated Text Detection - Deepfakes

##### Step 2: Finetuning small models

In [1]:
import torch

# Check if PyTorch can see the GPU
print("CUDA available:", torch.cuda.is_available())

# Print the GPU name if found
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti


In [2]:
# Fine-tune smaller models such as Ettin-68m on ChatGPT Abstracts dataset.
# Smaller dataset means fastest run which equals to 10-20mins
import pandas as pd
from datasets import Dataset

# Load the ChatGPT Abstracts dataset
train_dataset = pd.read_csv("data_splits/Chatgpt-Research-Abstracts_train.csv")
validation_dataset = pd.read_csv("data_splits/Chatgpt-Research-Abstracts_val.csv")
test_dataset = pd.read_csv("data_splits/Chatgpt-Research-Abstracts_test.csv")

# Converting the pandas tables into HuggingFace dataset format
train_ds = Dataset.from_pandas(train_dataset)
validation_ds = Dataset.from_pandas(validation_dataset)
test_ds = Dataset.from_pandas(test_dataset)

# print the sizes of the datasets
print("Train dataset size:", len(train_ds))
print("Validation dataset size:", len(validation_ds))
print("Test dataset size:", len(test_ds))

Train dataset size: 14000
Validation dataset size: 3000
Test dataset size: 3000


In [4]:
# load the model and tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# The model finetuning is ettin-encoder
model_name = "jhu-clsp/ettin-encoder-68m"

# Tokenizer converts text into tokens that the model can understand
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model for sequence classification (human=0, AI=1)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/118 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-encoder-68m
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# Tokenize the text
# Convert all texts into tokens, cutt off at 512 tokens.
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

train_ds = train_ds.map(tokenize, batched=True)
validation_ds = validation_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [6]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# compute accuracy and F1 score for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # pick the class with the highest score
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

# Training settings
training_args = TrainingArguments(
    output_dir="models/ettin68m_abstracts",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=32,   # batch size for evaluation
     learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",            # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=10,
    report_to="none"
)   

# Pads each batch to equal length automatically, so that the model can process it. This is important for variable-length sequences.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create the Trainer
trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_ds,              # training dataset
    eval_dataset=validation_ds,          # evaluation dataset
    data_collator=data_collator,         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.047069,0.029253,0.992333,0.992295
2,0.000038,0.044076,0.990333,0.990252
3,0.000006,0.036041,0.991667,0.991653


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2625, training_loss=0.02245275115910114, metrics={'train_runtime': 1024.069, 'train_samples_per_second': 41.013, 'train_steps_per_second': 2.563, 'total_flos': 5081803926587136.0, 'train_loss': 0.02245275115910114, 'epoch': 3.0})

In [7]:
# Evaluate the fine-tuned model on the unseen test set
results = trainer.evaluate(test_ds)
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000006,0.023918,3,0.993333,0.993307


{'eval_loss': 0.023918401449918747, 'eval_accuracy': 0.9933333333333333, 'eval_f1': 0.9933065595716198}


In [8]:
# Save the fine-tuned model and tokenizer
model.save_pretrained("models/ettin68m_abstracts_final")
tokenizer.save_pretrained("models/ettin68m_abstracts_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/ettin68m_abstracts_final\\tokenizer_config.json',
 'models/ettin68m_abstracts_final\\tokenizer.json')

In [10]:
# Cross-domain evaluation (RQ2)
# Evaluate the abstracts-trained model on the other 3 test datasets (raid, wiki and Mage)
# Load the other datasets
raid_test_dataset = pd.read_csv("data_splits/RAID_test.csv")
wiki_test_dataset = pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")
mage_test_dataset = pd.read_csv("data_splits/MAGE_test.csv")

# Converting the pandas tables into HuggingFace dataset format
raid_test_ds = Dataset.from_pandas(raid_test_dataset).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(wiki_test_dataset).map(tokenize, batched=True)
mage_test_ds = Dataset.from_pandas(mage_test_dataset).map(tokenize, batched=True)

# Evaluate the trained model on each of the other datasets
print("RAID:", trainer.evaluate(raid_test_ds))
print("Wiki:", trainer.evaluate(wiki_test_ds))
print("Mage:", trainer.evaluate(mage_test_ds))


Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000006,3.741948,3,0.569660,0.244597


RAID: {'eval_loss': 3.741947650909424, 'eval_accuracy': 0.569659935031008, 'eval_f1': 0.24459685780365262}


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000006,1.335645,3,0.690311,0.551666


Wiki: {'eval_loss': 1.3356451988220215, 'eval_accuracy': 0.6903111111111111, 'eval_f1': 0.5516664521940549}


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000006,4.644769,3,0.484283,0.008515


Mage: {'eval_loss': 4.644769191741943, 'eval_accuracy': 0.4842834690670096, 'eval_f1': 0.008515313830517787}


In [ ]:
# Each results will be saved as .csv
# Build 4x4 matrix 
import os

def save_results_to_csv(model_name, trained_on, tested_on, results):
    # Create a one-row table with the results
    row = pd.DataFrame({
        'model_name': [model_name],
        'trained_on': [trained_on],
        'tested_on': [tested_on],
        'accuracy': [results['eval_accuracy']],
        'f1': [results['eval_f1']],
        'loss': [results['eval_loss']]
    })

    # Append to results and create if it doesn't exist
    file_exists = os.path.isfile('all_models_evaluation_results.csv')
    row.to_csv('all_models_evaluation_results.csv', mode='a', header=not file_exists, index=False)
    print("Saved:", model_name, trained_on, tested_on)


In [16]:
# Save the results of the fine-tuned model on the test datasets
save_results_to_csv("ettin-68m", "ChatGPT Abstracts", "ChatGPT Abstracts", results)
save_results_to_csv("ettin-68m", "ChatGPT Abstracts", "RAID", trainer.evaluate(raid_test_ds))
save_results_to_csv("ettin-68m", "ChatGPT Abstracts", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("ettin-68m", "ChatGPT Abstracts", "Mage", trainer.evaluate(mage_test_ds))   

Saved: ettin-68m ChatGPT Abstracts ChatGPT Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000006,3.741948,3,0.569660,0.244597


Saved: ettin-68m ChatGPT Abstracts RAID


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000006,1.335645,3,0.690311,0.551666


Saved: ettin-68m ChatGPT Abstracts Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000006,4.644769,3,0.484283,0.008515


Saved: ettin-68m ChatGPT Abstracts Mage


In [17]:
# Fine-tune Ettin-68m on GPT-Wiki-intro dataset.
# Load the GPT-Wiki-intro dataset train/val splits
GPT_Wiki_intro_train_dataset = pd.read_csv("data_splits/GPT-Wiki-intro_train.csv")
GPT_Wiki_intro_validation_dataset = pd.read_csv("data_splits/GPT-Wiki-intro_val.csv")

# Convert to HuggingFace dataset format
GPT_Wiki_intro_train_ds = Dataset.from_pandas(GPT_Wiki_intro_train_dataset).map(tokenize, batched=True)
GPT_Wiki_intro_validation_ds = Dataset.from_pandas(GPT_Wiki_intro_validation_dataset).map(tokenize, batched=True)

# Load a fresh instance of the model and tokenizer for fine-tuning on the new dataset
model_name = "jhu-clsp/ettin-encoder-68m"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Training settings
training_args = TrainingArguments(
    output_dir="models/ettin68m_wiki",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=32,   # batch size for evaluation
     learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",            # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=GPT_Wiki_intro_train_ds,              # training dataset
    eval_dataset=GPT_Wiki_intro_validation_ds,          # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the wiki-trained model
trainer.train()

Map:   0%|          | 0/210000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/118 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-encoder-68m
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.009446,0.001692,0.999622,0.999622
2,0.000000,0.002596,0.999533,0.999533
3,0.000000,0.001924,0.999756,0.999756


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=39375, training_loss=0.003522793122583486, metrics={'train_runtime': 11463.747, 'train_samples_per_second': 54.956, 'train_steps_per_second': 3.435, 'total_flos': 6.193603706081837e+16, 'train_loss': 0.003522793122583486, 'epoch': 3.0})

In [19]:
# save and evaluate on all 4 test sets
# Save the trained Wiki model
model.save_pretrained("models/ettin68m_wiki_final")

# Load the ChatGPT Abstracts test dataset for evaluation
abstracts_test_ds = pd.read_csv("data_splits/Chatgpt-Research-Abstracts_test.csv")
abstracts_test = Dataset.from_pandas(abstracts_test_ds).map(tokenize, batched=True)

# Evaluate on all 4 test sets and save the results
results_abstracts = trainer.evaluate(abstracts_test)
save_results_to_csv("ettin-68m", "GPT-Wiki-intro", "GPT-Wiki-Abstracts", results_abstracts)
results_raid = trainer.evaluate(raid_test_ds)
save_results_to_csv("ettin-68m", "GPT-Wiki-intro", "GPT-Wiki-Raid", results_raid)
results_wiki = trainer.evaluate(wiki_test_ds)
save_results_to_csv("ettin-68m", "GPT-Wiki-intro", "GPT-Wiki", results_wiki)
results_mage = trainer.evaluate(mage_test_ds)
save_results_to_csv("ettin-68m", "GPT-Wiki-intro", "GPT-Wiki-Mage", results_mage)

print("\nRow 2 of the matrix complete!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000000,5.778263,3,0.556667,0.573991


Saved: ettin-68m GPT-Wiki-intro GPT-Wiki-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000000,2.621487,3,0.775948,0.744051


Saved: ettin-68m GPT-Wiki-intro GPT-Wiki-Raid


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000000,0.001884,3,0.999667,0.999667


Saved: ettin-68m GPT-Wiki-intro GPT-Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000000,6.422482,3,0.454386,0.402784


Saved: ettin-68m GPT-Wiki-intro GPT-Wiki-Mage

Row 2 of the matrix complete!


In [20]:
# Fine-tune smaller models such as Ettin-68m on RAID dataset.
# Load the RAID dataset train/val splits
raid_train_dataset = pd.read_csv("data_splits/RAID_train.csv")
raid_validation_dataset = pd.read_csv("data_splits/RAID_val.csv")

# Convert to HuggingFace dataset format
raid_train_ds = Dataset.from_pandas(raid_train_dataset).map(tokenize, batched=True)
raid_validation_ds = Dataset.from_pandas(raid_validation_dataset).map(tokenize, batched=True)

# Load a fresh instance of the model and tokenizer for fine-tuning on the new dataset
model_name = "jhu-clsp/ettin-encoder-68m"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Training settings
training_args = TrainingArguments(
    output_dir="models/ettin68m_raid",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=32,   # batch size for evaluation
     learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",            # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    report_to="none"
)

# Trainer for RAID dataset
trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=raid_train_ds,              # training dataset
    eval_dataset=raid_validation_ds,          # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the RAID-trained model
trainer.train()

Map:   0%|          | 0/205429 [00:00<?, ? examples/s]

Map:   0%|          | 0/44020 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/118 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-encoder-68m
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.068939,0.098740,0.973194,0.972766
2,0.040795,0.096076,0.977419,0.977105
3,0.019811,0.102437,0.982667,0.982605


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=38520, training_loss=0.06541618906340618, metrics={'train_runtime': 15505.4042, 'train_samples_per_second': 39.747, 'train_steps_per_second': 2.484, 'total_flos': 8.065269698139562e+16, 'train_loss': 0.06541618906340618, 'epoch': 3.0})

In [27]:
abstracts_test_ds = pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")
wiki_test_ds = pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")
mage_test_ds = pd.read_csv("data_splits/MAGE_test.csv")

abstracts_test_ds = Dataset.from_pandas(abstracts_test_ds).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(wiki_test_ds).map(tokenize, batched=True)
mage_test_ds = Dataset.from_pandas(mage_test_ds).map(tokenize, batched=True)

print(type(abstracts_test_ds))

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

<class 'datasets.arrow_dataset.Dataset'>


In [28]:
# Save the trained RAID model
model.save_pretrained("models/ettin68m_raid_final")

# Evaluate on all 4 test sets and save the results
save_results_to_csv("ettin-68m", "RAID", "RAID", trainer.evaluate(raid_test_ds))
save_results_to_csv("ettin-68m", "RAID", "RAID-Chatgpt-Research-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("ettin-68m", "RAID", "RAID-GPT-Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("ettin-68m", "RAID", "RAID-MAGE", trainer.evaluate(mage_test_ds))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.019811,0.093252,3,0.977147,0.976841


Saved: ettin-68m RAID RAID


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.019811,4.039788,3,0.721667,0.781585


Saved: ettin-68m RAID RAID-Chatgpt-Research-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.019811,0.026478,3,0.995133,0.995139


Saved: ettin-68m RAID RAID-GPT-Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.019811,6.891794,3,0.248821,0.333692


Saved: ettin-68m RAID RAID-MAGE


In [ ]:
# print which sources belong to each label, 'src' column reveals if 0/1 mean human or machine
# This cell is to investigte the bug found - not-to rerun
print("Label 0 sources:", mage_test_dataset[mage_test_dataset["label"] == 0]["src"].unique()[:10])
print("Label 1 sources:", mage_test_dataset[mage_test_dataset["label"] == 1]["src"].unique()[:10])

Label 0 sources: <ArrowStringArray>
[             'tldr_machine_continuation_65B',
         'xsum_machine_continuation_bloom_7b',
           'wp_machine_continuation_bloom_7b',
 'yelp_machine_continuation_opt_iml_max_1.3b',
    'tldr_machine_continuation_gpt-3.5-trubo',
           'squad_machine_continuation_gpt_j',
              'roct_machine_continuation_65B',
         'tldr_machine_continuation_bloom_7b',
         'eli5_machine_continuation_opt_125m',
         'yelp_machine_continuation_gpt_neox']
Length: 10, dtype: str
Label 1 sources: <ArrowStringArray>
[   'yelp_human',     'cmv_human',   'squad_human',    'eli5_human',
 'sci_gen_human',      'wp_human',   'hswag_human',    'tldr_human',
    'xsum_human',    'roct_human']
Length: 10, dtype: str


In [6]:
# Re-evaluate the 3 saved models on the fixed MAGE test set

import os
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score

# Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}
    

mage_test_fixed = pd.read_csv("data_splits/MAGE_test.csv")

# Saved Models
saved_models = {
    "GPT-Wiki-Intro": "models/ettin68m_wiki_final",
    "RAID": "models/ettin68m_raid_final",
}

# Loop each model, evaluate on fixed MAGE and save the result
for trained_on, model_path in saved_models.items():
    tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/ettin-encoder-68m")
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=512)
    mage_test_fixed_ds = Dataset.from_pandas(mage_test_fixed).map(tokenize, batched=True)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=32, report_to="none"),
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    save_results_to_csv("ettin-68m", trained_on, "Mage", trainer.evaluate(mage_test_fixed_ds))

print("\nAll 2 re-evaluations done")

Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Accuracy,F1
No log,5.050204,0,0.551829,0.510017


Saved: ettin-68m GPT-Wiki-Intro Mage


Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Accuracy,F1
No log,1.616276,0,0.754215,0.781243


Saved: ettin-68m RAID Mage

All 2 re-evaluations done


In [8]:
# Fine-tune Ettin-68m on MAGE (row 4 of the matrix)
# Load the corrected MAGE train/val splits
mage_train_dataset = pd.read_csv("data_splits/MAGE_train.csv")
mage_validation_dataset = pd.read_csv("data_splits/MAGE_val.csv")

# Load the base tokenizer (shared by all Ettin-68m models)
tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/ettin-encoder-68m")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert to HuggingFace Format and tokenize
mage_train_ds = Dataset.from_pandas(mage_train_dataset).map(tokenize, batched=True)
mage_val_ds = Dataset.from_pandas(mage_validation_dataset).map(tokenize, batched=True)

# Load the fresh base model
model = AutoModelForSequenceClassification.from_pretrained("jhu-clsp/ettin-encoder-68m", num_labels=2)

# Training settings for MAGE
training_args = TrainingArguments(
    output_dir="models/ettin68m_mage",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,                 # number of training epochs
    per_device_train_batch_size=16,     # batch size for training
    per_device_eval_batch_size=32,      # batch size for evaluation
    learning_rate=2e-5,                 # learning rate
    eval_strategy="epoch",              # evaluate each epoch
    save_strategy="epoch",              # save each epoch          
    load_best_model_at_end=True,         # load the best model when finished training (default metric is loss)
    logging_steps=50,
    report_to="none",
)

# Train the MAGE dataset
trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=mage_train_ds,         # training dataset
    eval_dataset=mage_val_ds,            # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the MAGE-trained model
trainer.train()

Map:   0%|          | 0/130645 [00:00<?, ? examples/s]

Map:   0%|          | 0/27995 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/118 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-encoder-68m
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.163059,0.136955,0.949634,0.950131
2,0.075900,0.148366,0.962029,0.962070
3,0.008147,0.250509,0.963779,0.963635


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24498, training_loss=0.10095109310417147, metrics={'train_runtime': 9830.8459, 'train_samples_per_second': 39.868, 'train_steps_per_second': 2.492, 'total_flos': 5.031747895439824e+16, 'train_loss': 0.10095109310417147, 'epoch': 3.0})

In [11]:
# Evaluate the MAGE model on all datasets
model.save_pretrained("models/ettin68m_mage_final")
tokenizer.save_pretrained("models/ettin68m_mage_final")

# Load and tokenize the other sets
abstracts_test_ds = pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")
wiki_test_ds = pd.read_csv("data_splits/GPT-Wiki-Intro_test.csv")
raid_test_ds = pd.read_csv("data_splits/RAID_test.csv")

abstracts_test_ds = Dataset.from_pandas(abstracts_test_ds).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(wiki_test_ds).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(raid_test_ds).map(tokenize, batched=True)

# MAGE test set - correct labels
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)

# Evaluate on all 4 dataset and save - 4x4 matrix
save_results_to_csv("ettin68m", "MAGE", "Mage", trainer.evaluate(mage_test_ds))
save_results_to_csv("ettin68m", "MAGE", "MAGE-Chatgpt-Research-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("ettin68m", "MAGE", "MAGE-GPT-Wiki-Intro", trainer.evaluate(wiki_test_ds))
save_results_to_csv("ettin68m", "MAGE", "MAGE-RAID", trainer.evaluate(raid_test_ds))

print("\n4x4 Matrix complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

...

Saved: ettin68m MAGE MAGE-Chatgpt-Research-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.008147,0.257414,3,0.898822,0.903532


Saved: ettin68m MAGE MAGE-GPT-Wiki-Intro


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.008147,0.918101,3,0.771495,0.792491


Saved: ettin68m MAGE MAGE-RAID

4x4 Matrix complete


## Fine-tune ModernBERT-base model on 4 datasets

In [ ]:
## Fine-tune ModernBERT-base Model on all 4 datasets for comparison - 4x4 Matrix 
# ModernBERT-Base FineTune on ChatGPT Abstracts 
# Same method as Ettin68m, different base model
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score

# Load Abstracts train/val splits
abstracts_train_dataset = pd.read_csv("data_splits/ChatGPT-Research-Abstracts_train.csv")
abstracts_validation_dataset = pd.read_csv("data_splits/ChatGPT-Research-Abstracts_val.csv")

# Load ModernBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert and tokenize
abstracts_train_ds = Dataset.from_pandas(abstracts_train_dataset).map(tokenize, batched=True)
abstracts_val_ds = Dataset.from_pandas(abstracts_validation_dataset).map(tokenize, batched=True)

# Load fresh ModernBERT-base with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=2)

# compute accuracy and F1 score for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # pick the class with the highest score
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

# Training settings
training_args = TrainingArguments(
    output_dir="models/modernbert_abstracts",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,  # batch size for training - was 16, fits full in VRAM now
    per_device_eval_batch_size=16,   # batch size for evaluation - was 32
    learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",            # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=abstracts_train_ds,              # training dataset
    eval_dataset=abstracts_val_ds,          # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the abstracts-trained model
trainer.train()

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.044183,0.081312,0.983667,0.983407
2,0.024482,0.055158,0.988333,0.988212
3,0.000016,0.044911,0.992000,0.991989


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5250, training_loss=0.04072367077595637, metrics={'train_runtime': 1898.95, 'train_samples_per_second': 22.117, 'train_steps_per_second': 2.765, 'total_flos': 1.2190004966309184e+16, 'train_loss': 0.04072367077595637, 'epoch': 3.0})

In [ ]:
# Build 4x4 matrix
import os

def save_results_to_csv(model_name, trained_on, tested_on, results):
    # Create a one-row table with the results
    row = pd.DataFrame({
        'model_name': [model_name],
        'trained_on': [trained_on],
        'tested_on': [tested_on],
        'accuracy': [results['eval_accuracy']],
        'f1': [results['eval_f1']],
        'loss': [results['eval_loss']]
    })

    # Append to results and create if it doesn't exist
    file_exists = os.path.isfile('all_models_evaluation_results.csv')
    row.to_csv('all_models_evaluation_results.csv', mode='a', header=not file_exists, index=False)
    print("Saved:", model_name, trained_on, tested_on)

In [8]:
# Save the ModernBERT abstracts model and tokenizer
model.save_pretrained("models/modernbert_abstracts_final")
tokenizer.save_pretrained("models/modernbert_abstracts_final")

# Load and tokenize all 4 test sets with the ModernBERT tokenizer
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")).map(tokenize, batched=True)
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)

# Evaluate on all 4 and save
save_results_to_csv("modernbert-base", "ChatGPT Abstracts", "ChatGPT Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("modernbert-base", "ChatGPT Abstracts", "RAID", trainer.evaluate(raid_test_ds))
save_results_to_csv("modernbert-base", "ChatGPT Abstracts", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("modernbert-base", "ChatGPT Abstracts", "Mage", trainer.evaluate(mage_test_ds))

print("\nModernBERT row 1 complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

...

Saved: modernbert-base ChatGPT Abstracts RAID


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000016,2.776182,3,0.674133,0.517282


Saved: modernbert-base ChatGPT Abstracts Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000016,5.678299,3,0.530647,0.128994


Saved: modernbert-base ChatGPT Abstracts Mage

ModernBERT row 1 complete


In [ ]:
## Fine-tune ModernBERT-base Model on all 4 datasets for comparison - 4x4 Matrix 
# ModernBERT-Base FineTune on GPT-Wiki-Intro

# Load Wiki train/val splits
GPT_Wiki_intro_train_dataset = pd.read_csv("data_splits/GPT-Wiki-Intro_train.csv")
GPT_Wiki_intro_validation_dataset = pd.read_csv("data_splits/GPT-Wiki-Intro_val.csv")

# Load ModernBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert and tokenize
GPT_Wiki_intro_train_ds = Dataset.from_pandas(GPT_Wiki_intro_train_dataset).map(tokenize, batched=True)
GPT_Wiki_intro_validation_ds = Dataset.from_pandas(GPT_Wiki_intro_validation_dataset).map(tokenize, batched=True)

# Load fresh ModernBERT-base with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=2)

# compute accuracy and F1 score for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # pick the class with the highest score
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

# Training settings
training_args = TrainingArguments(
    output_dir="models/modernbert_wiki",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,  # batch size for training - was 16, fits full in VRAM now
    per_device_eval_batch_size=16,   # batch size for evaluation - was 32
    learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",            # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=GPT_Wiki_intro_train_ds,              # training dataset
    eval_dataset=GPT_Wiki_intro_validation_ds,          # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the wiki-trained model
trainer.train()

Map:   0%|          | 0/210000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.000068,0.004836,0.999133,0.999133
2,0.000003,0.004287,0.999289,0.999288
3,0.000000,0.003845,0.999511,0.999511


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=78750, training_loss=0.00462938153934376, metrics={'train_runtime': 22432.2169, 'train_samples_per_second': 28.085, 'train_steps_per_second': 3.511, 'total_flos': 1.4683977565515686e+17, 'train_loss': 0.00462938153934376, 'epoch': 3.0})

In [11]:
# Save the ModernBERT Wiki model and tokenizer
model.save_pretrained("models/modernbert_wiki_final")
tokenizer.save_pretrained("models/modernbert_wiki_final")

# Load and tokenize all 4 test sets with the ModernBERT tokenizer
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")).map(tokenize, batched=True)
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)

# Evaluate on all 4 and save
save_results_to_csv("modernbert-base", "GPT-Wiki-Intro", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("modernbert-base", "GPT-Wiki-Intro", "ChatGPT-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("modernbert-base", "GPT-Wiki-Intro", "RAID", trainer.evaluate(raid_test_ds))
save_results_to_csv("modernbert-base", "GPT-Wiki-Intro", "MAGE", trainer.evaluate(mage_test_ds))

print("\nModernBERT row 2 complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

...

Saved: modernbert-base GPT-Wiki-Intro ChatGPT-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000000,3.698307,3,0.753231,0.712155


Saved: modernbert-base GPT-Wiki-Intro RAID


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000000,7.089345,3,0.514895,0.528552


Saved: modernbert-base GPT-Wiki-Intro MAGE

ModernBERT row 2 complete


In [1]:
# import all necessary libaries
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
## Fine-tune ModernBERT-base Model on all 4 datasets for comparison - 4x4 Matrix 
# ModernBERT-Base FineTune on RAID

# Load RAID train/val splits
raid_train_dataset = pd.read_csv("data_splits/RAID_train.csv")
raid_validation_dataset = pd.read_csv("data_splits/RAID_val.csv")

# Load ModernBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert and tokenize
raid_train_ds = Dataset.from_pandas(raid_train_dataset).map(tokenize, batched=True)
raid_validation_ds = Dataset.from_pandas(raid_validation_dataset).map(tokenize, batched=True)

# Load fresh ModernBERT-base with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=2)

# compute accuracy and F1 score for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # pick the class with the highest score
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

# Training settings
training_args = TrainingArguments(
    output_dir="models/modernbert_raid",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,  # batch size for training - was 16, fits full in VRAM now
    per_device_eval_batch_size=16,   # batch size for evaluation - was 32
    learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",            # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    bf16=True,                         # mixed precision for faster and less memory on RTX 5060 Ti
    report_to="none"
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=raid_train_ds,              # training dataset
    eval_dataset=raid_validation_ds,          # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the RAID-trained model
trainer.train()

Map:   0%|          | 0/205429 [00:00<?, ? examples/s]

Map:   0%|          | 0/44020 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.108225,0.077415,0.975852,0.975586
2,0.037860,0.079540,0.979350,0.979370
3,0.038496,0.094581,0.982485,0.982417


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=77037, training_loss=0.07266062610938913, metrics={'train_runtime': 14888.1421, 'train_samples_per_second': 41.394, 'train_steps_per_second': 5.174, 'total_flos': 2.081311280372426e+17, 'train_loss': 0.07266062610938913, 'epoch': 3.0})

In [ ]:
# Build 4x4 matrix
import os

def save_results_to_csv(model_name, trained_on, tested_on, results):
    # Create a one-row table with the results
    row = pd.DataFrame({
        'model_name': [model_name],
        'trained_on': [trained_on],
        'tested_on': [tested_on],
        'accuracy': [results['eval_accuracy']],
        'f1': [results['eval_f1']],
        'loss': [results['eval_loss']]
    })

    # Append to results and create if it doesn't exist
    file_exists = os.path.isfile('all_models_evaluation_results.csv')
    row.to_csv('all_models_evaluation_results.csv', mode='a', header=not file_exists, index=False)
    print("Saved:", model_name, trained_on, tested_on)

In [ ]:
# Save the ModernBERT RAID model and tokenizer
model.save_pretrained("models/modernbert_raid_final")
tokenizer.save_pretrained("models/modernbert_raid_final")

# Load and tokenize all 4 test sets with ModernBERT tokenizer
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-Intro_test.csv")).map(tokenize, batched=True)
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)

# Evaluate each — label matches the dataset on the same line
save_results_to_csv("modernbert-base", "RAID", "RAID", trainer.evaluate(raid_test_ds))
save_results_to_csv("modernbert-base", "RAID", "ChatGPT-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("modernbert-base", "RAID", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("modernbert-base", "RAID", "MAGE", trainer.evaluate(mage_test_ds))

print("\nModernBERT row 3 complete")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

...

Saved: modernbert-base RAID ChatGPT-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.038496,0.041716,3,0.990044,0.990092


Saved: modernbert-base RAID Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.038496,1.241570,3,0.751464,0.782983


Saved: modernbert-base RAID MAGE

ModernBERT row 3 complete


In [7]:
## Fine-tune ModernBERT-base Model on all 4 datasets for comparison - 4x4 Matrix 
# ModernBERT-Base FineTune on MAGE

# Load MAGE train/val splits
mage_train_dataset = pd.read_csv("data_splits/MAGE_train.csv")
mage_validation_dataset = pd.read_csv("data_splits/MAGE_val.csv")

# Load ModernBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert and tokenize
mage_train_ds = Dataset.from_pandas(mage_train_dataset).map(tokenize, batched=True)
mage_validation_ds = Dataset.from_pandas(mage_validation_dataset).map(tokenize, batched=True)

# Load fresh ModernBERT-base with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=2)

# compute accuracy and F1 score for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # pick the class with the highest score
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

# Training settings
training_args = TrainingArguments(
    output_dir="models/modernbert_mage",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,  # batch size for training - was 16, fits full in VRAM now
    per_device_eval_batch_size=16,   # batch size for evaluation - was 32
    learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",            # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    bf16=True,                         # mixed precision for faster and less memory on RTX 5060 Ti
    report_to="none"
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=mage_train_ds,              # training dataset
    eval_dataset=mage_validation_ds,          # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the MAGE-trained model
trainer.train()

Map:   0%|          | 0/130645 [00:00<?, ? examples/s]

Map:   0%|          | 0/27995 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.261942,0.139149,0.954135,0.953660
2,0.124228,0.144794,0.965065,0.964856
3,0.042185,0.197902,0.968066,0.967959


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=48993, training_loss=0.11206151672699004, metrics={'train_runtime': 8759.505, 'train_samples_per_second': 44.744, 'train_steps_per_second': 5.593, 'total_flos': 1.2076495133533648e+17, 'train_loss': 0.11206151672699004, 'epoch': 3.0})

In [8]:
# Save the ModernBERT MAGE model and tokenizer
model.save_pretrained("models/modernbert_mage_final")
tokenizer.save_pretrained("models/modernbert_mage_final")

# Load and tokenize all 4 test sets with ModernBERT tokenizer
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-Intro_test.csv")).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)

# Evaluate each — label matches the dataset on the same line
save_results_to_csv("modernbert-base", "MAGE", "MAGE", trainer.evaluate(mage_test_ds))
save_results_to_csv("modernbert-base", "MAGE", "ChatGPT-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("modernbert-base", "MAGE", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("modernbert-base", "MAGE", "RAID", trainer.evaluate(raid_test_ds))

print("\nModernBERT row 4 complete - BOth 4x4 matrices done")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

...

Saved: modernbert-base MAGE ChatGPT-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.042185,0.314192,3,0.895156,0.893834


Saved: modernbert-base MAGE Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.042185,1.037392,3,0.767225,0.766216


Saved: modernbert-base MAGE RAID

ModernBERT row 4 complete - BOth 4x4 matrices done
